# Seasonal Periods in Time Series

This notebook explains **seasonal periods** using Markdown notes, formulas, Python examples, and plots.

## Learning goals

By the end, you should be able to:

- explain what a seasonal period is;
- distinguish **data frequency** from **seasonal period**;
- calculate seasonal periods for common time intervals;
- understand multiple seasonalities;
- use seasonal lags and simple lag correlations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. What is a seasonal period?

A **seasonal period** is the number of observations before a seasonal pattern repeats.

If the seasonal period is \(m\), observations separated by \(m\) steps may behave similarly.

Examples:

- monthly data with yearly seasonality: \(m=12\)
- daily data with weekly seasonality: \(m=7\)
- hourly data with daily seasonality: \(m=24\)

> The seasonal period is measured in **number of observations**.

## 2. Common seasonal periods

| Data frequency | Repeating pattern | Seasonal period |
|---|---|---:|
| Quarterly | Yearly | 4 |
| Monthly | Yearly | 12 |
| Weekly | Yearly | about 52 |
| Daily | Weekly | 7 |
| Daily | Yearly | about 365.25 |
| Hourly | Daily | 24 |
| Hourly | Weekly | 168 |
| Hourly | Yearly | 8766 |
| Minute | Hourly | 60 |
| Minute | Daily | 1440 |
| Minute | Weekly | 10080 |
| Minute | Yearly | 525960 |

A useful idea is:

$$
\text{seasonal period}
=
\frac{\text{length of seasonal cycle}}
{\text{time between observations}}
$$


## 3. Monthly data example

Suppose sales are recorded once per month and the pattern repeats every year.

There are 12 monthly observations in one year:

$$
m=12
$$


In [ ]:
dates = pd.date_range("2022-01-01", periods=48, freq="MS")
t = np.arange(48)

seasonal = 20 * np.sin(2 * np.pi * t / 12)
trend = 0.5 * t
noise = np.random.normal(0, 3, 48)

sales = 100 + trend + seasonal + noise

monthly_df = pd.DataFrame({
    "date": dates,
    "sales": sales
})

monthly_df.head()

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(monthly_df["date"], monthly_df["sales"], marker="o")
plt.title("Synthetic Monthly Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.grid(alpha=0.3)
plt.show()

The pattern repeats approximately every 12 observations, although trend and noise prevent exact repetition.

In [ ]:
monthly_df["sales_lag_12"] = monthly_df["sales"].shift(12)
monthly_df["difference_from_last_year"] = (
    monthly_df["sales"] - monthly_df["sales_lag_12"]
)

monthly_df.tail(18)

## 4. Frequency vs seasonal period

**Frequency** tells us how often observations are recorded.

**Seasonal period** tells us how many observations form one repeating cycle.

For hourly data with a daily cycle:

$$
24 \text{ observations/day}
$$

so:

$$
m=24
$$

For hourly data with a weekly cycle:

$$
24\times7=168
$$

so:

$$
m=168
$$


In [ ]:
print("Hourly data, daily seasonality:", 24)
print("Hourly data, weekly seasonality:", 24 * 7)
print("Hourly data, yearly seasonality:", 24 * 365.25)

## 5. Daily data can have multiple seasonalities

Daily data may contain:

### Weekly seasonality

$$
m=7
$$

### Annual seasonality

$$
m\approx365.25
$$

So one time series can contain more than one repeating seasonal pattern.


In [ ]:
dates = pd.date_range("2024-01-01", periods=730, freq="D")
t = np.arange(len(dates))

weekly = 8 * np.sin(2 * np.pi * t / 7)
annual = 15 * np.sin(2 * np.pi * t / 365.25)
trend = 0.02 * t
noise = np.random.normal(0, 2.5, len(t))

daily_sales = 100 + trend + weekly + annual + noise

daily_df = pd.DataFrame({
    "date": dates,
    "sales": daily_sales
})

daily_df.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(daily_df["date"], daily_df["sales"])
plt.title("Daily Data with Weekly and Annual Seasonality")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.grid(alpha=0.3)
plt.show()

Over two years, the long annual movement is visible. To make weekly repetition easier to see, zoom into a few weeks.

In [ ]:
sample = daily_df.iloc[:42]

plt.figure(figsize=(12, 4))
plt.plot(sample["date"], sample["sales"], marker="o")
plt.title("First 6 Weeks: Weekly Seasonality")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.show()

## 6. Minute-level data

If observations are recorded every minute:

- hourly seasonality: \(60\)
- daily seasonality: \(60\times24=1440\)
- weekly seasonality: \(60\times24\times7=10080\)
- yearly seasonality: \(60\times24\times365.25=525960\)

In [ ]:
minute_periods = pd.Series({
    "hourly": 60,
    "daily": 60 * 24,
    "weekly": 60 * 24 * 7,
    "yearly": 60 * 24 * 365.25
}, name="seasonal_period")

minute_periods

## 7. Why 52 weeks is only an approximation

An average year is approximately 365.25 days.

$$
\frac{365.25}{7}\approx52.18
$$

So yearly seasonality in weekly data is not exactly 52.

Many models require an integer seasonal period, so \(52\) is often used as a practical approximation.


In [ ]:
weeks_per_year = 365.25 / 7

print("Average weeks per year:", weeks_per_year)
print("Rounded value:", round(weeks_per_year))

## 8. Seasonal lags

A seasonal lag compares each observation with an observation one seasonal cycle earlier.

Examples:

- daily data with weekly seasonality → lag 7
- monthly data with yearly seasonality → lag 12
- hourly data with daily seasonality → lag 24

In pandas, use `.shift()`.

In [ ]:
example = pd.DataFrame({
    "value": [10, 12, 13, 9, 11, 18, 20, 11, 13, 14, 10, 12, 19, 21]
})

example["lag_7"] = example["value"].shift(7)
example

For daily data, `lag_7` aligns each day with the corresponding day one week earlier.

## 9. Lag correlation

If a seasonal pattern is strong, the series may be strongly related to a lagged copy of itself.

For monthly yearly seasonality, lag 12 is especially interesting.

For daily weekly seasonality, lag 7 is especially interesting.

In [ ]:
def lag_correlation(series, lag):
    return series.corr(series.shift(lag))

print(
    "Monthly correlation at lag 12:",
    round(lag_correlation(monthly_df["sales"], 12), 3)
)

print(
    "Daily correlation at lag 7:",
    round(lag_correlation(daily_df["sales"], 7), 3)
)

In [ ]:
lags = range(1, 25)
corrs = [lag_correlation(monthly_df["sales"], lag) for lag in lags]

plt.figure(figsize=(10, 4))
plt.stem(list(lags), corrs)
plt.axvline(12, linestyle="--", label="Lag 12")
plt.title("Lag Correlations for Monthly Data")
plt.xlabel("Lag")
plt.ylabel("Correlation")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

This is a simple diagnostic. In real time-series analysis, seasonality may also be studied using autocorrelation functions, decomposition methods, spectral methods, or model-based approaches.

## 10. Multiple seasonalities in hourly data

Hourly electricity demand might contain all of these:

| Pattern | Seasonal period |
|---|---:|
| Daily | 24 |
| Weekly | 168 |
| Yearly | 8766 |

This is called **multiple seasonality**.

Specialized models such as **MSTL**, **MFLES**, and **TBATS** can be useful for more complicated seasonal structures.

## 11. Helper function

In [ ]:
def common_seasonal_periods(frequency):
    periods = {
        "quarterly": {"yearly": 4},
        "monthly": {"yearly": 12},
        "weekly": {"yearly_approx": 52},
        "daily": {"weekly": 7, "yearly_approx": 365.25},
        "hourly": {
            "daily": 24,
            "weekly": 24 * 7,
            "yearly_approx": 24 * 365.25,
        },
        "minute": {
            "hourly": 60,
            "daily": 60 * 24,
            "weekly": 60 * 24 * 7,
            "yearly_approx": 60 * 24 * 365.25,
        },
    }

    frequency = frequency.lower()

    if frequency not in periods:
        raise ValueError(f"Choose from: {list(periods)}")

    return periods[frequency]


common_seasonal_periods("hourly")

## 12. Practice questions

### Question 1

You have monthly revenue data with a yearly cycle. What is \(m\)?

### Question 2

Website traffic is recorded hourly and has a daily pattern. What is \(m\)?

### Question 3

Electricity consumption is recorded every 30 minutes and has a daily pattern. What is \(m\)?

### Question 4

Daily sales show both weekday effects and yearly effects. Which seasonal periods might you consider?

In [ ]:
# Answers

q1 = 12
q2 = 24
q3 = 2 * 24
q4 = {"weekly": 7, "yearly_approx": 365.25}

print("Q1:", q1)
print("Q2:", q2)
print("Q3:", q3)
print("Q4:", q4)

## 13. Final mental model

Whenever you need a seasonal period:

1. Identify how often the data is observed.
2. Identify the cycle you expect to repeat.
3. Count how many observations occur in that cycle.

Example: hourly data with weekly seasonality:

$$
24\times7=168
$$

Therefore:

$$
\boxed{m=168}
$$

> **Frequency tells you how often observations are recorded. Seasonal period tells you how many observations occur before a seasonal pattern repeats.**
